# SwingTradingEnv v2 — Colab Notebook (with Data Harness & Walk‑Forward)

This notebook follows your documentation exactly:
- **Reward**: realized cash PnL / (prev_equity × risk_cap_pct), **clipped** to [-1, +1]
- **Costs**: spread + fees on notional delta; **funding** from `fund_vwap`, prorated 8h → 5m (long **pays**, short **earns**)
- **Liveness**: when `feed_alive` (or `depth_available==0`) → **force 1×**, **block new risk**, **exit/hold allowed**
- **Action space**: **MultiDiscrete([4,4,5])**, explicit masks; no flip‑in‑step; tighten‑only stops; add‑only leverage
- **Leak audit**: `fund_ts`, `liq_ts`, `basis_ts` ≤ `ts` (current & next step checked)

It includes:
- Phase‑4 style **data harness** checks
- **Walk‑forward** splits (Train=2021, Val=2022, Test=2023, Live tail=2024+ when present)
- **MaskablePPO** training on 2021 and evaluation on 2022/2023 with metrics (Sharpe, MaxDD, Calmar, Turnover, Stop‑hit proxy)

In [ ]:
# Dependencies for Colab runtime
!pip -q install --upgrade pip
!pip -q install "gymnasium>=0.29.1" "stable-baselines3[extra]>=2.3.0" "sb3-contrib>=2.3.0" \
                "numpy>=1.24" "pandas>=2.0" "pyarrow>=15.0" "matplotlib>=3.8" "pytest>=8.2" "tqdm>=4.66"

In [ ]:
# Mount Google Drive (in Colab). Safe no-op outside Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted.")
except Exception as e:
    print("Colab drive mount not available or already mounted:", e)

In [ ]:
# Paths provided by you
LOG_PATH   = "/content/drive/MyDrive/Crypto_Data/logs/build_token_v1_csv.log"
META_PATH  = "/content/drive/MyDrive/Crypto_Data/processed/btc_token_meta.parquet"
TOKEN_ALL  = "/content/drive/MyDrive/Crypto_Data/processed/btc_token_v1_5m.parquet"
TOKEN_L90  = "/content/drive/MyDrive/Crypto_Data/processed/btc_token_v1_5m_last90d.parquet"
MANIFEST   = "/content/drive/MyDrive/Crypto_Data/processed/token_v1_manifest.json"

RAW_BIN    = "/content/drive/MyDrive/Crypto_Data/raw/coinglass/btc_binance_funding_h4.csv"
RAW_BYB    = "/content/drive/MyDrive/Crypto_Data/raw/coinglass/btc_bybit_funding_h4.csv"
RAW_OKX    = "/content/drive/MyDrive/Crypto_Data/raw/coinglass/btc_okx_funding_h4.csv"
RAW_F_4H   = "/content/drive/MyDrive/Crypto_Data/raw/coinglass/btc_funding_oiw_4h.csv"
RAW_LIQ_1D = "/content/drive/MyDrive/Crypto_Data/raw/coinglass/btc_liq_1d.csv"
RAW_LIQ_4H = "/content/drive/MyDrive/Crypto_Data/raw/coinglass/btc_liq_4h.csv"
RAW_PERP_1M= "/content/drive/MyDrive/Crypto_Data/raw/perp/btc_perp_ohlcv_1m.csv"
RAW_SPOT_1M= "/content/drive/MyDrive/Crypto_Data/raw/spot/btc_spot_ohlcv_1m.csv"
BUILDER_PY = "/content/drive/MyDrive/Crypto_Data/build_token_v1_csv.py"

## Environment (exact spec)

In [ ]:
# ── swing_trading_env_v2.py ────────────────────────────────────────────────────
from __future__ import annotations
import numpy as np
import pandas as pd
from typing import Tuple, List, Optional, Dict, Any
import warnings

import gymnasium as gym
from gymnasium import spaces

TOKEN_COLUMNS: List[str] = [
    "log_ret_open","log_ret_high","log_ret_low","log_ret_close","range_frac",
    "log_volume","volume_zscore","depth_10bp","bid_ask_imbalance",
    "fund_binance","fund_bybit","fund_okx","fund_vwap","fund_zscore",
    "fund_spread_bin_byb","fund_spread_width","fund_flip","fund_accum_24h",
    "perp_spot_basis",
    "liq_buy_1d","liq_buy_6h","liq_buy_4h","liq_sell_1d","liq_sell_6h","liq_sell_4h",
    "vol_of_vol","sin_time","cos_time",
]

def _assert_token_contract(df: pd.DataFrame):
    required = ["ts","px","fund_vwap","fund_ts","liq_ts","basis_ts","feed_alive"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Data missing required columns: {missing}")
    for c in TOKEN_COLUMNS:
        if c not in df.columns:
            warnings.warn(f"Column '{c}' missing in data; filling zeros (development backfill).")
            df[c] = 0.0
    if not np.issubdtype(df["ts"].dtype, np.datetime64):
        df["ts"] = pd.to_datetime(df["ts"], utc=True)
    else:
        df["ts"] = df["ts"].dt.tz_localize("UTC") if df["ts"].dt.tz is None else df["ts"].dt.tz_convert("UTC")
    if not df["ts"].is_monotonic_increasing:
        raise AssertionError("Timestamps must be strictly increasing.")
    ms = (df["ts"].astype("int64") // 1_000_000).to_numpy()
    if np.any(ms % 300_000 != 0):
        raise AssertionError("Timestamps must align to 5-minute anchors (:00,:05,:10,...) UTC.")
    for c in ("fund_ts","liq_ts","basis_ts"):
        if not np.issubdtype(df[c].dtype, np.datetime64):
            df[c] = pd.to_datetime(df[c], utc=True)
        else:
            df[c] = df[c].dt.tz_localize("UTC") if df[c].dt.tz is None else df[c].dt.tz_convert("UTC")
    df["feed_alive"] = df["feed_alive"].astype(int)

def _flatten_idx(op: int, lev: int, stp: int) -> int:
    return op * 20 + lev * 5 + stp

def _unflatten_idx(idx: int) -> Tuple[int,int,int]:
    op = idx // 20
    r = idx % 20
    lev = r // 5
    stp = r % 5
    return op, lev, stp

class SwingTradingEnv:
    metadata: Dict[str, Any] = {"render.modes": []}
    def __init__(self, cfg: Dict[str, Any]):
        self.cfg = cfg.copy()
        self.data: pd.DataFrame = cfg["data"].copy()
        _assert_token_contract(self.data)
        self.initial_equity: float = float(cfg.get("initial_equity", 10_000.0))
        self.risk_cap_pct: float = float(cfg.get("risk_cap_pct", 0.05))
        self.reward_clip: Tuple[float,float] = tuple(cfg.get("reward_clip", (-1.0, 1.0)))
        self.max_leverage: int = int(cfg.get("max_leverage", 5))
        self.leverage_slots: List[int] = list(cfg.get("leverage_slots",[1,2,3,5]))
        self.stop_slots: List[float] = list(cfg.get("stop_slots",[0.001,0.002,0.003,0.005,0.008]))
        self.spread: float = float(cfg.get("spread", 0.0))
        self.fee_pct: float = float(cfg.get("fee_pct", 0.0))
        self.funding_model: str = str(cfg.get("funding_model","8h_to_5m"))
        self.random_start: bool = bool(cfg.get("random_start", True))
        self.episode_len: int = int(cfg.get("episode_len", 2400))
        self.if_train: bool = bool(cfg.get("if_train", True))
        self.seed_value: int = int(cfg.get("seed", 1337))
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(28,), dtype=np.float32)
        self.action_space = spaces.MultiDiscrete(np.array([4,4,5], dtype=np.int64))
        self.rng = np.random.default_rng(self.seed_value)
        self.ptr: int = 0
        self.start_ptr: int = 0
        self.end_ptr: int = min(self.episode_len, len(self.data)-1)
        self.equity: float = self.initial_equity
        self.pos_dir: int = 0
        self.lev_idx: int = 0
        self.stop_idx: int = 0
        self.entry_px: Optional[float] = None

    def _encode_obs(self, idx: int) -> np.ndarray:
        idx = max(0, min(idx, len(self.data)-1))
        row = self.data.iloc[idx]
        return row[TOKEN_COLUMNS].to_numpy(dtype=np.float32)

    def _legal_triples(self, row: pd.Series, blind: bool) -> List[Tuple[int,int,int]]:
        legal: List[Tuple[int,int,int]] = []
        if blind:
            legal.append((0,0,0))
            legal.append((3,0,0))
            return legal
        legal.append((0,0,0))
        legal.append((3,0,0))
        if self.pos_dir == 0:
            for lev in range(len(self.leverage_slots)):
                for stp in range(len(self.stop_slots)):
                    legal.append((1,lev,stp))
                    legal.append((2,lev,stp))
        elif self.pos_dir > 0:
            for lev in range(self.lev_idx, len(self.leverage_slots)):
                for stp in range(0, self.stop_idx+1):
                    legal.append((1,lev,stp))
        else:
            for lev in range(self.lev_idx, len(self.leverage_slots)):
                for stp in range(0, self.stop_idx+1):
                    legal.append((2,lev,stp))
        return legal

    def _coerce_to_legal(self, action: np.ndarray, legal: List[Tuple[int,int,int]]) -> Tuple[int,int,int]:
        a = (int(action[0]), int(action[1]), int(action[2]))
        if a in legal:
            return a
        return (0,0,0)

    def _position_notional(self, ref_equity: float) -> float:
        if self.pos_dir == 0:
            return 0.0
        lev = self.leverage_slots[self.lev_idx]
        return float(ref_equity) * float(lev)

    def _funding_per_bar(self, eight_hour_rate: float) -> float:
        if self.funding_model != "8h_to_5m":
            raise ValueError("Only funding_model='8h_to_5m' is supported.")
        return float(eight_hour_rate) / 96.0

    def reset(self):
        n = len(self.data)
        if self.random_start and self.if_train:
            max_start = max(0, n - (self.episode_len + 1))
            self.start_ptr = int(self.rng.integers(0, max_start+1)) if max_start > 0 else 0
        else:
            self.start_ptr = 0
        self.ptr = self.start_ptr
        self.end_ptr = min(self.start_ptr + self.episode_len, n-1)
        self.equity = self.initial_equity
        self.pos_dir = 0
        self.lev_idx = 0
        self.stop_idx = 0
        self.entry_px = None
        row = self.data.iloc[self.ptr]
        if (row["fund_ts"] > row["ts"]) or (row["liq_ts"] > row["ts"]) or (row["basis_ts"] > row["ts"]):
            raise AssertionError("Slow-feed timestamp later than bar time at reset.")
        return self._encode_obs(self.ptr)

    def step(self, action: np.ndarray):
        row = self.data.iloc[self.ptr]
        prev_idx = max(0, self.ptr-1)
        prev_row = self.data.iloc[prev_idx]
        blind = (int(row.get("feed_alive", 1)) == 0) or (int(row.get("depth_available", 1)) == 0)
        legal = self._legal_triples(row, blind)
        op, lev_sel, stp_sel = self._coerce_to_legal(action, legal)
        prev_equity = self.equity
        cash_delta = 0.0
        if self.pos_dir != 0:
            px_prev = float(prev_row["px"])
            px_now  = float(row["px"])
            if px_prev <= 0 or px_now <= 0:
                raise AssertionError("Non-positive price encountered.")
            ret = (px_now / px_prev) - 1.0
            notional = self._position_notional(prev_equity)
            price_pnl = float(self.pos_dir) * notional * ret
            cash_delta += price_pnl
            eight_hr = float(row["fund_vwap"])
            per_bar = self._funding_per_bar(eight_hr)
            carry = -abs(notional) * per_bar if self.pos_dir > 0 else +abs(notional) * per_bar
            cash_delta += carry
        old_notional = self._position_notional(prev_equity)
        if op == 3:
            new_pos_dir, new_lev_idx, new_stop_idx = 0, 0, 0
        elif op == 1:
            new_pos_dir = +1 if self.pos_dir == 0 else self.pos_dir
            new_lev_idx = lev_sel if self.pos_dir == 0 else max(self.lev_idx, lev_sel)
            new_stop_idx = stp_sel if self.pos_dir == 0 else min(self.stop_idx, stp_sel)
        elif op == 2:
            new_pos_dir = -1 if self.pos_dir == 0 else self.pos_dir
            new_lev_idx = lev_sel if self.pos_dir == 0 else max(self.lev_idx, lev_sel)
            new_stop_idx = stp_sel if self.pos_dir == 0 else min(self.stop_idx, stp_sel)
        else:
            new_pos_dir, new_lev_idx, new_stop_idx = self.pos_dir, self.lev_idx, self.stop_idx
        if self.pos_dir > 0 and op == 2:
            new_pos_dir, new_lev_idx, new_stop_idx = self.pos_dir, self.lev_idx, self.stop_idx
        if self.pos_dir < 0 and op == 1:
            new_pos_dir, new_lev_idx, new_stop_idx = self.pos_dir, self.lev_idx, self.stop_idx
        new_notional = 0.0 if new_pos_dir == 0 else float(prev_equity) * float(self.leverage_slots[new_lev_idx])
        trade_delta = new_notional - old_notional
        fill_cost = abs(trade_delta) * (abs(self.fee_pct) + abs(self.spread))
        cash_delta -= fill_cost
        self.equity = float(self.equity + cash_delta)
        self.pos_dir = new_pos_dir
        self.lev_idx = int(new_lev_idx)
        self.stop_idx = int(new_stop_idx)
        self.entry_px = float(row["px"]) if (self.pos_dir != 0 and (op in (1,2) or old_notional == 0)) else self.entry_px
        r_unit = max(prev_equity * self.risk_cap_pct, 1e-6)
        reward = cash_delta / r_unit
        reward = float(np.clip(reward, self.reward_clip[0], self.reward_clip[1]))
        self.ptr += 1
        done = bool(self.ptr >= self.end_ptr or self.ptr >= (len(self.data)-1))
        if self.ptr < len(self.data):
            nrow = self.data.iloc[self.ptr]
            if (nrow["fund_ts"] > nrow["ts"]) or (nrow["liq_ts"] > nrow["ts"]) or (nrow["basis_ts"] > nrow["ts"]):
                raise AssertionError("Slow-feed timestamp later than bar time at next step.")
        obs = self._encode_obs(self.ptr if self.ptr < len(self.data) else len(self.data)-1)
        info = {"pnl": cash_delta, "equity": self.equity, "prev_equity": prev_equity}
        return obs, reward, done, info

    def legal_actions_flat(self) -> List[int]:
        row = self.data.iloc[self.ptr]
        blind = (int(row.get("feed_alive",1)) == 0) or (int(row.get("depth_available",1)) == 0)
        triples = self._legal_triples(row, blind)
        return [_flatten_idx(*t) for t in triples]

class SB3Adapter(gym.Env):
    metadata = {"render_modes": []}
    def __init__(self, base_env: SwingTradingEnv):
        super().__init__()
        self.base = base_env
        self.action_space = self.base.action_space
        self.observation_space = self.base.observation_space
    def reset(self, *, seed: Optional[int] = None, options: Optional[dict] = None):
        if seed is not None:
            self.base.rng = np.random.default_rng(int(seed))
        obs = self.base.reset()
        info = {}
        return obs, info
    def step(self, action):
        obs, reward, done, info = self.base.step(action)
        terminated = bool(done)
        truncated = False
        return obs, float(reward), terminated, truncated, info

def _flatten_idx(op: int, lev: int, stp: int) -> int:
    return op * 20 + lev * 5 + stp

def _unflatten_idx(idx: int) -> Tuple[int,int,int]:
    op = idx // 20
    r = idx % 20
    lev = r // 5
    stp = r % 5
    return op, lev, stp

class FlattenActionWrapper(gym.Wrapper):
    def __init__(self, env: gym.Env):
        super().__init__(env)
        assert isinstance(env.action_space, spaces.MultiDiscrete), "Underlying env must be MultiDiscrete."
        self.action_space = spaces.Discrete(80)
    def reset(self, **kwargs):
        return self.env.reset(**kwargs)
    def step(self, action_idx: int):
        op, lev, stp = _unflatten_idx(int(action_idx))
        return self.env.step(np.array([op,lev,stp], dtype=np.int64))
    def get_action_mask_80(self) -> np.ndarray:
        legal = self.env.base.legal_actions_flat() if isinstance(self.env, SB3Adapter) else self.env.legal_actions_flat()
        mask = np.zeros(80, dtype=bool)
        mask[np.array(legal, dtype=int)] = True
        return mask

In [ ]:
# Write environment module
with open("swing"\n# \u2500\u2500 swing_trading_env_v2.py \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nfrom __future__ import annotations\nimport numpy as np\nimport pandas as pd\nfrom typing import Tuple, List, Optional, Dict, Any\nimport warnings\n\nimport gymnasium as gym\nfrom gymnasium import spaces\n\nTOKEN_COLUMNS: List[str] = [\n    \"log_ret_open\",\"log_ret_high\",\"log_ret_low\",\"log_ret_close\",\"range_frac\",\n    \"log_volume\",\"volume_zscore\",\"depth_10bp\",\"bid_ask_imbalance\",\n    \"fund_binance\",\"fund_bybit\",\"fund_okx\",\"fund_vwap\",\"fund_zscore\",\n    \"fund_spread_bin_byb\",\"fund_spread_width\",\"fund_flip\",\"fund_accum_24h\",\n    \"perp_spot_basis\",\n    \"liq_buy_1d\",\"liq_buy_6h\",\"liq_buy_4h\",\"liq_sell_1d\",\"liq_sell_6h\",\"liq_sell_4h\",\n    \"vol_of_vol\",\"sin_time\",\"cos_time\",\n]\n\ndef _assert_token_contract(df: pd.DataFrame):\n    required = [\"ts\",\"px\",\"fund_vwap\",\"fund_ts\",\"liq_ts\",\"basis_ts\",\"feed_alive\"]\n    missing = [c for c in required if c not in df.columns]\n    if missing:\n        raise ValueError(f\"Data missing required columns: {missing}\")\n    for c in TOKEN_COLUMNS:\n        if c not in df.columns:\n            warnings.warn(f\"Column '{c}' missing in data; filling zeros (development backfill).\")\n            df[c] = 0.0\n    if not np.issubdtype(df[\"ts\"].dtype, np.datetime64):\n        df[\"ts\"] = pd.to_datetime(df[\"ts\"], utc=True)\n    else:\n        df[\"ts\"] = df[\"ts\"].dt.tz_localize(\"UTC\") if df[\"ts\"].dt.tz is None else df[\"ts\"].dt.tz_convert(\"UTC\")\n    if not df[\"ts\"].is_monotonic_increasing:\n        raise AssertionError(\"Timestamps must be strictly increasing.\")\n    ms = (df[\"ts\"].astype(\"int64\") // 1_000_000).to_numpy()\n    if np.any(ms % 300_000 != 0):\n        raise AssertionError(\"Timestamps must align to 5-minute anchors (:00,:05,:10,...) UTC.\")\n    for c in (\"fund_ts\",\"liq_ts\",\"basis_ts\"):\n        if not np.issubdtype(df[c].dtype, np.datetime64):\n            df[c] = pd.to_datetime(df[c], utc=True)\n        else:\n            df[c] = df[c].dt.tz_localize(\"UTC\") if df[c].dt.tz is None else df[c].dt.tz_convert(\"UTC\")\n    df[\"feed_alive\"] = df[\"feed_alive\"].astype(int)\n\ndef _flatten_idx(op: int, lev: int, stp: int) -> int:\n    return op * 20 + lev * 5 + stp\n\ndef _unflatten_idx(idx: int) -> Tuple[int,int,int]:\n    op = idx // 20\n    r = idx % 20\n    lev = r // 5\n    stp = r % 5\n    return op, lev, stp\n\nclass SwingTradingEnv:\n    metadata: Dict[str, Any] = {\"render.modes\": []}\n    def __init__(self, cfg: Dict[str, Any]):\n        self.cfg = cfg.copy()\n        self.data: pd.DataFrame = cfg[\"data\"].copy()\n        _assert_token_contract(self.data)\n        self.initial_equity: float = float(cfg.get(\"initial_equity\", 10_000.0))\n        self.risk_cap_pct: float = float(cfg.get(\"risk_cap_pct\", 0.05))\n        self.reward_clip: Tuple[float,float] = tuple(cfg.get(\"reward_clip\", (-1.0, 1.0)))\n        self.max_leverage: int = int(cfg.get(\"max_leverage\", 5))\n        self.leverage_slots: List[int] = list(cfg.get(\"leverage_slots\",[1,2,3,5]))\n        self.stop_slots: List[float] = list(cfg.get(\"stop_slots\",[0.001,0.002,0.003,0.005,0.008]))\n        self.spread: float = float(cfg.get(\"spread\", 0.0))\n        self.fee_pct: float = float(cfg.get(\"fee_pct\", 0.0))\n        self.funding_model: str = str(cfg.get(\"funding_model\",\"8h_to_5m\"))\n        self.random_start: bool = bool(cfg.get(\"random_start\", True))\n        self.episode_len: int = int(cfg.get(\"episode_len\", 2400))\n        self.if_train: bool = bool(cfg.get(\"if_train\", True))\n        self.seed_value: int = int(cfg.get(\"seed\", 1337))\n        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(28,), dtype=np.float32)\n        self.action_space = spaces.MultiDiscrete(np.array([4,4,5], dtype=np.int64))\n        self.rng = np.random.default_rng(self.seed_value)\n        self.ptr: int = 0\n        self.start_ptr: int = 0\n        self.end_ptr: int = min(self.episode_len, len(self.data)-1)\n        self.equity: float = self.initial_equity\n        self.pos_dir: int = 0\n        self.lev_idx: int = 0\n        self.stop_idx: int = 0\n        self.entry_px: Optional[float] = None\n\n    def _encode_obs(self, idx: int) -> np.ndarray:\n        idx = max(0, min(idx, len(self.data)-1))\n        row = self.data.iloc[idx]\n        return row[TOKEN_COLUMNS].to_numpy(dtype=np.float32)\n\n    def _legal_triples(self, row: pd.Series, blind: bool) -> List[Tuple[int,int,int]]:\n        legal: List[Tuple[int,int,int]] = []\n        if blind:\n            legal.append((0,0,0))\n            legal.append((3,0,0))\n            return legal\n        legal.append((0,0,0))\n        legal.append((3,0,0))\n        if self.pos_dir == 0:\n            for lev in range(len(self.leverage_slots)):\n                for stp in range(len(self.stop_slots)):\n                    legal.append((1,lev,stp))\n                    legal.append((2,lev,stp))\n        elif self.pos_dir > 0:\n            for lev in range(self.lev_idx, len(self.leverage_slots)):\n                for stp in range(0, self.stop_idx+1):\n                    legal.append((1,lev,stp))\n        else:\n            for lev in range(self.lev_idx, len(self.leverage_slots)):\n                for stp in range(0, self.stop_idx+1):\n                    legal.append((2,lev,stp))\n        return legal\n\n    def _coerce_to_legal(self, action: np.ndarray, legal: List[Tuple[int,int,int]]) -> Tuple[int,int,int]:\n        a = (int(action[0]), int(action[1]), int(action[2]))\n        if a in legal:\n            return a\n        return (0,0,0)\n\n    def _position_notional(self, ref_equity: float) -> float:\n        if self.pos_dir == 0:\n            return 0.0\n        lev = self.leverage_slots[self.lev_idx]\n        return float(ref_equity) * float(lev)\n\n    def _funding_per_bar(self, eight_hour_rate: float) -> float:\n        if self.funding_model != \"8h_to_5m\":\n            raise ValueError(\"Only funding_model='8h_to_5m' is supported.\")\n        return float(eight_hour_rate) / 96.0\n\n    def reset(self):\n        n = len(self.data)\n        if self.random_start and self.if_train:\n            max_start = max(0, n - (self.episode_len + 1))\n            self.start_ptr = int(self.rng.integers(0, max_start+1)) if max_start > 0 else 0\n        else:\n            self.start_ptr = 0\n        self.ptr = self.start_ptr\n        self.end_ptr = min(self.start_ptr + self.episode_len, n-1)\n        self.equity = self.initial_equity\n        self.pos_dir = 0\n        self.lev_idx = 0\n        self.stop_idx = 0\n        self.entry_px = None\n        row = self.data.iloc[self.ptr]\n        if (row[\"fund_ts\"] > row[\"ts\"]) or (row[\"liq_ts\"] > row[\"ts\"]) or (row[\"basis_ts\"] > row[\"ts\"]):\n            raise AssertionError(\"Slow-feed timestamp later than bar time at reset.\")\n        return self._encode_obs(self.ptr)\n\n    def step(self, action: np.ndarray):\n        row = self.data.iloc[self.ptr]\n        prev_idx = max(0, self.ptr-1)\n        prev_row = self.data.iloc[prev_idx]\n        blind = (int(row.get(\"feed_alive\", 1)) == 0) or (int(row.get(\"depth_available\", 1)) == 0)\n        legal = self._legal_triples(row, blind)\n        op, lev_sel, stp_sel = self._coerce_to_legal(action, legal)\n        prev_equity = self.equity\n        cash_delta = 0.0\n        if self.pos_dir != 0:\n            px_prev = float(prev_row[\"px\"])\n            px_now  = float(row[\"px\"])\n            if px_prev <= 0 or px_now <= 0:\n                raise AssertionError(\"Non-positive price encountered.\")\n            ret = (px_now / px_prev) - 1.0\n            notional = self._position_notional(prev_equity)\n            price_pnl = float(self.pos_dir) * notional * ret\n            cash_delta += price_pnl\n            eight_hr = float(row[\"fund_vwap\"])\n            per_bar = self._funding_per_bar(eight_hr)\n            carry = -abs(notional) * per_bar if self.pos_dir > 0 else +abs(notional) * per_bar\n            cash_delta += carry\n        old_notional = self._position_notional(prev_equity)\n        if op == 3:\n            new_pos_dir, new_lev_idx, new_stop_idx = 0, 0, 0\n        elif op == 1:\n            new_pos_dir = +1 if self.pos_dir == 0 else self.pos_dir\n            new_lev_idx = lev_sel if self.pos_dir == 0 else max(self.lev_idx, lev_sel)\n            new_stop_idx = stp_sel if self.pos_dir == 0 else min(self.stop_idx, stp_sel)\n        elif op == 2:\n            new_pos_dir = -1 if self.pos_dir == 0 else self.pos_dir\n            new_lev_idx = lev_sel if self.pos_dir == 0 else max(self.lev_idx, lev_sel)\n            new_stop_idx = stp_sel if self.pos_dir == 0 else min(self.stop_idx, stp_sel)\n        else:\n            new_pos_dir, new_lev_idx, new_stop_idx = self.pos_dir, self.lev_idx, self.stop_idx\n        if self.pos_dir > 0 and op == 2:\n            new_pos_dir, new_lev_idx, new_stop_idx = self.pos_dir, self.lev_idx, self.stop_idx\n        if self.pos_dir < 0 and op == 1:\n            new_pos_dir, new_lev_idx, new_stop_idx = self.pos_dir, self.lev_idx, self.stop_idx\n        new_notional = 0.0 if new_pos_dir == 0 else float(prev_equity) * float(self.leverage_slots[new_lev_idx])\n        trade_delta = new_notional - old_notional\n        fill_cost = abs(trade_delta) * (abs(self.fee_pct) + abs(self.spread))\n        cash_delta -= fill_cost\n        self.equity = float(self.equity + cash_delta)\n        self.pos_dir = new_pos_dir\n        self.lev_idx = int(new_lev_idx)\n        self.stop_idx = int(new_stop_idx)\n        self.entry_px = float(row[\"px\"]) if (self.pos_dir != 0 and (op in (1,2) or old_notional == 0)) else self.entry_px\n        r_unit = max(prev_equity * self.risk_cap_pct, 1e-6)\n        reward = cash_delta / r_unit\n        reward = float(np.clip(reward, self.reward_clip[0], self.reward_clip[1]))\n        self.ptr += 1\n        done = bool(self.ptr >= self.end_ptr or self.ptr >= (len(self.data)-1))\n        if self.ptr < len(self.data):\n            nrow = self.data.iloc[self.ptr]\n            if (nrow[\"fund_ts\"] > nrow[\"ts\"]) or (nrow[\"liq_ts\"] > nrow[\"ts\"]) or (nrow[\"basis_ts\"] > nrow[\"ts\"]):\n                raise AssertionError(\"Slow-feed timestamp later than bar time at next step.\")\n        obs = self._encode_obs(self.ptr if self.ptr < len(self.data) else len(self.data)-1)\n        info = {\"pnl\": cash_delta, \"equity\": self.equity, \"prev_equity\": prev_equity}\n        return obs, reward, done, info\n\n    def legal_actions_flat(self) -> List[int]:\n        row = self.data.iloc[self.ptr]\n        blind = (int(row.get(\"feed_alive\",1)) == 0) or (int(row.get(\"depth_available\",1)) == 0)\n        triples = self._legal_triples(row, blind)\n        return [_flatten_idx(*t) for t in triples]\n\nclass SB3Adapter(gym.Env):\n    metadata = {\"render_modes\": []}\n    def __init__(self, base_env: SwingTradingEnv):\n        super().__init__()\n        self.base = base_env\n        self.action_space = self.base.action_space\n        self.observation_space = self.base.observation_space\n    def reset(self, *, seed: Optional[int] = None, options: Optional[dict] = None):\n        if seed is not None:\n            self.base.rng = np.random.default_rng(int(seed))\n        obs = self.base.reset()\n        info = {}\n        return obs, info\n    def step(self, action):\n        obs, reward, done, info = self.base.step(action)\n        terminated = bool(done)\n        truncated = False\n        return obs, float(reward), terminated, truncated, info\n\ndef _flatten_idx(op: int, lev: int, stp: int) -> int:\n    return op * 20 + lev * 5 + stp\n\ndef _unflatten_idx(idx: int) -> Tuple[int,int,int]:\n    op = idx // 20\n    r = idx % 20\n    lev = r // 5\n    stp = r % 5\n    return op, lev, stp\n\nclass FlattenActionWrapper(gym.Wrapper):\n    def __init__(self, env: gym.Env):\n        super().__init__(env)\n        assert isinstance(env.action_space, spaces.MultiDiscrete), \"Underlying env must be MultiDiscrete.\"\n        self.action_space = spaces.Discrete(80)\n    def reset(self, **kwargs):\n        return self.env.reset(**kwargs)\n    def step(self, action_idx: int):\n        op, lev, stp = _unflatten_idx(int(action_idx))\n        return self.env.step(np.array([op,lev,stp], dtype=np.int64))\n    def get_action_mask_80(self) -> np.ndarray:\n        legal = self.env.base.legal_actions_flat() if isinstance(self.env, SB3Adapter) else self.env.legal_actions_flat()\n        mask = np.zeros(80, dtype=bool)\n        mask[np.array(legal, dtype=int)] = True\n        return mask\n"trading"\n# \u2500\u2500 swing_trading_env_v2.py \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nfrom __future__ import annotations\nimport numpy as np\nimport pandas as pd\nfrom typing import Tuple, List, Optional, Dict, Any\nimport warnings\n\nimport gymnasium as gym\nfrom gymnasium import spaces\n\nTOKEN_COLUMNS: List[str] = [\n    \"log_ret_open\",\"log_ret_high\",\"log_ret_low\",\"log_ret_close\",\"range_frac\",\n    \"log_volume\",\"volume_zscore\",\"depth_10bp\",\"bid_ask_imbalance\",\n    \"fund_binance\",\"fund_bybit\",\"fund_okx\",\"fund_vwap\",\"fund_zscore\",\n    \"fund_spread_bin_byb\",\"fund_spread_width\",\"fund_flip\",\"fund_accum_24h\",\n    \"perp_spot_basis\",\n    \"liq_buy_1d\",\"liq_buy_6h\",\"liq_buy_4h\",\"liq_sell_1d\",\"liq_sell_6h\",\"liq_sell_4h\",\n    \"vol_of_vol\",\"sin_time\",\"cos_time\",\n]\n\ndef _assert_token_contract(df: pd.DataFrame):\n    required = [\"ts\",\"px\",\"fund_vwap\",\"fund_ts\",\"liq_ts\",\"basis_ts\",\"feed_alive\"]\n    missing = [c for c in required if c not in df.columns]\n    if missing:\n        raise ValueError(f\"Data missing required columns: {missing}\")\n    for c in TOKEN_COLUMNS:\n        if c not in df.columns:\n            warnings.warn(f\"Column '{c}' missing in data; filling zeros (development backfill).\")\n            df[c] = 0.0\n    if not np.issubdtype(df[\"ts\"].dtype, np.datetime64):\n        df[\"ts\"] = pd.to_datetime(df[\"ts\"], utc=True)\n    else:\n        df[\"ts\"] = df[\"ts\"].dt.tz_localize(\"UTC\") if df[\"ts\"].dt.tz is None else df[\"ts\"].dt.tz_convert(\"UTC\")\n    if not df[\"ts\"].is_monotonic_increasing:\n        raise AssertionError(\"Timestamps must be strictly increasing.\")\n    ms = (df[\"ts\"].astype(\"int64\") // 1_000_000).to_numpy()\n    if np.any(ms % 300_000 != 0):\n        raise AssertionError(\"Timestamps must align to 5-minute anchors (:00,:05,:10,...) UTC.\")\n    for c in (\"fund_ts\",\"liq_ts\",\"basis_ts\"):\n        if not np.issubdtype(df[c].dtype, np.datetime64):\n            df[c] = pd.to_datetime(df[c], utc=True)\n        else:\n            df[c] = df[c].dt.tz_localize(\"UTC\") if df[c].dt.tz is None else df[c].dt.tz_convert(\"UTC\")\n    df[\"feed_alive\"] = df[\"feed_alive\"].astype(int)\n\ndef _flatten_idx(op: int, lev: int, stp: int) -> int:\n    return op * 20 + lev * 5 + stp\n\ndef _unflatten_idx(idx: int) -> Tuple[int,int,int]:\n    op = idx // 20\n    r = idx % 20\n    lev = r // 5\n    stp = r % 5\n    return op, lev, stp\n\nclass SwingTradingEnv:\n    metadata: Dict[str, Any] = {\"render.modes\": []}\n    def __init__(self, cfg: Dict[str, Any]):\n        self.cfg = cfg.copy()\n        self.data: pd.DataFrame = cfg[\"data\"].copy()\n        _assert_token_contract(self.data)\n        self.initial_equity: float = float(cfg.get(\"initial_equity\", 10_000.0))\n        self.risk_cap_pct: float = float(cfg.get(\"risk_cap_pct\", 0.05))\n        self.reward_clip: Tuple[float,float] = tuple(cfg.get(\"reward_clip\", (-1.0, 1.0)))\n        self.max_leverage: int = int(cfg.get(\"max_leverage\", 5))\n        self.leverage_slots: List[int] = list(cfg.get(\"leverage_slots\",[1,2,3,5]))\n        self.stop_slots: List[float] = list(cfg.get(\"stop_slots\",[0.001,0.002,0.003,0.005,0.008]))\n        self.spread: float = float(cfg.get(\"spread\", 0.0))\n        self.fee_pct: float = float(cfg.get(\"fee_pct\", 0.0))\n        self.funding_model: str = str(cfg.get(\"funding_model\",\"8h_to_5m\"))\n        self.random_start: bool = bool(cfg.get(\"random_start\", True))\n        self.episode_len: int = int(cfg.get(\"episode_len\", 2400))\n        self.if_train: bool = bool(cfg.get(\"if_train\", True))\n        self.seed_value: int = int(cfg.get(\"seed\", 1337))\n        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(28,), dtype=np.float32)\n        self.action_space = spaces.MultiDiscrete(np.array([4,4,5], dtype=np.int64))\n        self.rng = np.random.default_rng(self.seed_value)\n        self.ptr: int = 0\n        self.start_ptr: int = 0\n        self.end_ptr: int = min(self.episode_len, len(self.data)-1)\n        self.equity: float = self.initial_equity\n        self.pos_dir: int = 0\n        self.lev_idx: int = 0\n        self.stop_idx: int = 0\n        self.entry_px: Optional[float] = None\n\n    def _encode_obs(self, idx: int) -> np.ndarray:\n        idx = max(0, min(idx, len(self.data)-1))\n        row = self.data.iloc[idx]\n        return row[TOKEN_COLUMNS].to_numpy(dtype=np.float32)\n\n    def _legal_triples(self, row: pd.Series, blind: bool) -> List[Tuple[int,int,int]]:\n        legal: List[Tuple[int,int,int]] = []\n        if blind:\n            legal.append((0,0,0))\n            legal.append((3,0,0))\n            return legal\n        legal.append((0,0,0))\n        legal.append((3,0,0))\n        if self.pos_dir == 0:\n            for lev in range(len(self.leverage_slots)):\n                for stp in range(len(self.stop_slots)):\n                    legal.append((1,lev,stp))\n                    legal.append((2,lev,stp))\n        elif self.pos_dir > 0:\n            for lev in range(self.lev_idx, len(self.leverage_slots)):\n                for stp in range(0, self.stop_idx+1):\n                    legal.append((1,lev,stp))\n        else:\n            for lev in range(self.lev_idx, len(self.leverage_slots)):\n                for stp in range(0, self.stop_idx+1):\n                    legal.append((2,lev,stp))\n        return legal\n\n    def _coerce_to_legal(self, action: np.ndarray, legal: List[Tuple[int,int,int]]) -> Tuple[int,int,int]:\n        a = (int(action[0]), int(action[1]), int(action[2]))\n        if a in legal:\n            return a\n        return (0,0,0)\n\n    def _position_notional(self, ref_equity: float) -> float:\n        if self.pos_dir == 0:\n            return 0.0\n        lev = self.leverage_slots[self.lev_idx]\n        return float(ref_equity) * float(lev)\n\n    def _funding_per_bar(self, eight_hour_rate: float) -> float:\n        if self.funding_model != \"8h_to_5m\":\n            raise ValueError(\"Only funding_model='8h_to_5m' is supported.\")\n        return float(eight_hour_rate) / 96.0\n\n    def reset(self):\n        n = len(self.data)\n        if self.random_start and self.if_train:\n            max_start = max(0, n - (self.episode_len + 1))\n            self.start_ptr = int(self.rng.integers(0, max_start+1)) if max_start > 0 else 0\n        else:\n            self.start_ptr = 0\n        self.ptr = self.start_ptr\n        self.end_ptr = min(self.start_ptr + self.episode_len, n-1)\n        self.equity = self.initial_equity\n        self.pos_dir = 0\n        self.lev_idx = 0\n        self.stop_idx = 0\n        self.entry_px = None\n        row = self.data.iloc[self.ptr]\n        if (row[\"fund_ts\"] > row[\"ts\"]) or (row[\"liq_ts\"] > row[\"ts\"]) or (row[\"basis_ts\"] > row[\"ts\"]):\n            raise AssertionError(\"Slow-feed timestamp later than bar time at reset.\")\n        return self._encode_obs(self.ptr)\n\n    def step(self, action: np.ndarray):\n        row = self.data.iloc[self.ptr]\n        prev_idx = max(0, self.ptr-1)\n        prev_row = self.data.iloc[prev_idx]\n        blind = (int(row.get(\"feed_alive\", 1)) == 0) or (int(row.get(\"depth_available\", 1)) == 0)\n        legal = self._legal_triples(row, blind)\n        op, lev_sel, stp_sel = self._coerce_to_legal(action, legal)\n        prev_equity = self.equity\n        cash_delta = 0.0\n        if self.pos_dir != 0:\n            px_prev = float(prev_row[\"px\"])\n            px_now  = float(row[\"px\"])\n            if px_prev <= 0 or px_now <= 0:\n                raise AssertionError(\"Non-positive price encountered.\")\n            ret = (px_now / px_prev) - 1.0\n            notional = self._position_notional(prev_equity)\n            price_pnl = float(self.pos_dir) * notional * ret\n            cash_delta += price_pnl\n            eight_hr = float(row[\"fund_vwap\"])\n            per_bar = self._funding_per_bar(eight_hr)\n            carry = -abs(notional) * per_bar if self.pos_dir > 0 else +abs(notional) * per_bar\n            cash_delta += carry\n        old_notional = self._position_notional(prev_equity)\n        if op == 3:\n            new_pos_dir, new_lev_idx, new_stop_idx = 0, 0, 0\n        elif op == 1:\n            new_pos_dir = +1 if self.pos_dir == 0 else self.pos_dir\n            new_lev_idx = lev_sel if self.pos_dir == 0 else max(self.lev_idx, lev_sel)\n            new_stop_idx = stp_sel if self.pos_dir == 0 else min(self.stop_idx, stp_sel)\n        elif op == 2:\n            new_pos_dir = -1 if self.pos_dir == 0 else self.pos_dir\n            new_lev_idx = lev_sel if self.pos_dir == 0 else max(self.lev_idx, lev_sel)\n            new_stop_idx = stp_sel if self.pos_dir == 0 else min(self.stop_idx, stp_sel)\n        else:\n            new_pos_dir, new_lev_idx, new_stop_idx = self.pos_dir, self.lev_idx, self.stop_idx\n        if self.pos_dir > 0 and op == 2:\n            new_pos_dir, new_lev_idx, new_stop_idx = self.pos_dir, self.lev_idx, self.stop_idx\n        if self.pos_dir < 0 and op == 1:\n            new_pos_dir, new_lev_idx, new_stop_idx = self.pos_dir, self.lev_idx, self.stop_idx\n        new_notional = 0.0 if new_pos_dir == 0 else float(prev_equity) * float(self.leverage_slots[new_lev_idx])\n        trade_delta = new_notional - old_notional\n        fill_cost = abs(trade_delta) * (abs(self.fee_pct) + abs(self.spread))\n        cash_delta -= fill_cost\n        self.equity = float(self.equity + cash_delta)\n        self.pos_dir = new_pos_dir\n        self.lev_idx = int(new_lev_idx)\n        self.stop_idx = int(new_stop_idx)\n        self.entry_px = float(row[\"px\"]) if (self.pos_dir != 0 and (op in (1,2) or old_notional == 0)) else self.entry_px\n        r_unit = max(prev_equity * self.risk_cap_pct, 1e-6)\n        reward = cash_delta / r_unit\n        reward = float(np.clip(reward, self.reward_clip[0], self.reward_clip[1]))\n        self.ptr += 1\n        done = bool(self.ptr >= self.end_ptr or self.ptr >= (len(self.data)-1))\n        if self.ptr < len(self.data):\n            nrow = self.data.iloc[self.ptr]\n            if (nrow[\"fund_ts\"] > nrow[\"ts\"]) or (nrow[\"liq_ts\"] > nrow[\"ts\"]) or (nrow[\"basis_ts\"] > nrow[\"ts\"]):\n                raise AssertionError(\"Slow-feed timestamp later than bar time at next step.\")\n        obs = self._encode_obs(self.ptr if self.ptr < len(self.data) else len(self.data)-1)\n        info = {\"pnl\": cash_delta, \"equity\": self.equity, \"prev_equity\": prev_equity}\n        return obs, reward, done, info\n\n    def legal_actions_flat(self) -> List[int]:\n        row = self.data.iloc[self.ptr]\n        blind = (int(row.get(\"feed_alive\",1)) == 0) or (int(row.get(\"depth_available\",1)) == 0)\n        triples = self._legal_triples(row, blind)\n        return [_flatten_idx(*t) for t in triples]\n\nclass SB3Adapter(gym.Env):\n    metadata = {\"render_modes\": []}\n    def __init__(self, base_env: SwingTradingEnv):\n        super().__init__()\n        self.base = base_env\n        self.action_space = self.base.action_space\n        self.observation_space = self.base.observation_space\n    def reset(self, *, seed: Optional[int] = None, options: Optional[dict] = None):\n        if seed is not None:\n            self.base.rng = np.random.default_rng(int(seed))\n        obs = self.base.reset()\n        info = {}\n        return obs, info\n    def step(self, action):\n        obs, reward, done, info = self.base.step(action)\n        terminated = bool(done)\n        truncated = False\n        return obs, float(reward), terminated, truncated, info\n\ndef _flatten_idx(op: int, lev: int, stp: int) -> int:\n    return op * 20 + lev * 5 + stp\n\ndef _unflatten_idx(idx: int) -> Tuple[int,int,int]:\n    op = idx // 20\n    r = idx % 20\n    lev = r // 5\n    stp = r % 5\n    return op, lev, stp\n\nclass FlattenActionWrapper(gym.Wrapper):\n    def __init__(self, env: gym.Env):\n        super().__init__(env)\n        assert isinstance(env.action_space, spaces.MultiDiscrete), \"Underlying env must be MultiDiscrete.\"\n        self.action_space = spaces.Discrete(80)\n    def reset(self, **kwargs):\n        return self.env.reset(**kwargs)\n    def step(self, action_idx: int):\n        op, lev, stp = _unflatten_idx(int(action_idx))\n        return self.env.step(np.array([op,lev,stp], dtype=np.int64))\n    def get_action_mask_80(self) -> np.ndarray:\n        legal = self.env.base.legal_actions_flat() if isinstance(self.env, SB3Adapter) else self.env.legal_actions_flat()\n        mask = np.zeros(80, dtype=bool)\n        mask[np.array(legal, dtype=int)] = True\n        return mask\n"env"\n# \u2500\u2500 swing_trading_env_v2.py \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nfrom __future__ import annotations\nimport numpy as np\nimport pandas as pd\nfrom typing import Tuple, List, Optional, Dict, Any\nimport warnings\n\nimport gymnasium as gym\nfrom gymnasium import spaces\n\nTOKEN_COLUMNS: List[str] = [\n    \"log_ret_open\",\"log_ret_high\",\"log_ret_low\",\"log_ret_close\",\"range_frac\",\n    \"log_volume\",\"volume_zscore\",\"depth_10bp\",\"bid_ask_imbalance\",\n    \"fund_binance\",\"fund_bybit\",\"fund_okx\",\"fund_vwap\",\"fund_zscore\",\n    \"fund_spread_bin_byb\",\"fund_spread_width\",\"fund_flip\",\"fund_accum_24h\",\n    \"perp_spot_basis\",\n    \"liq_buy_1d\",\"liq_buy_6h\",\"liq_buy_4h\",\"liq_sell_1d\",\"liq_sell_6h\",\"liq_sell_4h\",\n    \"vol_of_vol\",\"sin_time\",\"cos_time\",\n]\n\ndef _assert_token_contract(df: pd.DataFrame):\n    required = [\"ts\",\"px\",\"fund_vwap\",\"fund_ts\",\"liq_ts\",\"basis_ts\",\"feed_alive\"]\n    missing = [c for c in required if c not in df.columns]\n    if missing:\n        raise ValueError(f\"Data missing required columns: {missing}\")\n    for c in TOKEN_COLUMNS:\n        if c not in df.columns:\n            warnings.warn(f\"Column '{c}' missing in data; filling zeros (development backfill).\")\n            df[c] = 0.0\n    if not np.issubdtype(df[\"ts\"].dtype, np.datetime64):\n        df[\"ts\"] = pd.to_datetime(df[\"ts\"], utc=True)\n    else:\n        df[\"ts\"] = df[\"ts\"].dt.tz_localize(\"UTC\") if df[\"ts\"].dt.tz is None else df[\"ts\"].dt.tz_convert(\"UTC\")\n    if not df[\"ts\"].is_monotonic_increasing:\n        raise AssertionError(\"Timestamps must be strictly increasing.\")\n    ms = (df[\"ts\"].astype(\"int64\") // 1_000_000).to_numpy()\n    if np.any(ms % 300_000 != 0):\n        raise AssertionError(\"Timestamps must align to 5-minute anchors (:00,:05,:10,...) UTC.\")\n    for c in (\"fund_ts\",\"liq_ts\",\"basis_ts\"):\n        if not np.issubdtype(df[c].dtype, np.datetime64):\n            df[c] = pd.to_datetime(df[c], utc=True)\n        else:\n            df[c] = df[c].dt.tz_localize(\"UTC\") if df[c].dt.tz is None else df[c].dt.tz_convert(\"UTC\")\n    df[\"feed_alive\"] = df[\"feed_alive\"].astype(int)\n\ndef _flatten_idx(op: int, lev: int, stp: int) -> int:\n    return op * 20 + lev * 5 + stp\n\ndef _unflatten_idx(idx: int) -> Tuple[int,int,int]:\n    op = idx // 20\n    r = idx % 20\n    lev = r // 5\n    stp = r % 5\n    return op, lev, stp\n\nclass SwingTradingEnv:\n    metadata: Dict[str, Any] = {\"render.modes\": []}\n    def __init__(self, cfg: Dict[str, Any]):\n        self.cfg = cfg.copy()\n        self.data: pd.DataFrame = cfg[\"data\"].copy()\n        _assert_token_contract(self.data)\n        self.initial_equity: float = float(cfg.get(\"initial_equity\", 10_000.0))\n        self.risk_cap_pct: float = float(cfg.get(\"risk_cap_pct\", 0.05))\n        self.reward_clip: Tuple[float,float] = tuple(cfg.get(\"reward_clip\", (-1.0, 1.0)))\n        self.max_leverage: int = int(cfg.get(\"max_leverage\", 5))\n        self.leverage_slots: List[int] = list(cfg.get(\"leverage_slots\",[1,2,3,5]))\n        self.stop_slots: List[float] = list(cfg.get(\"stop_slots\",[0.001,0.002,0.003,0.005,0.008]))\n        self.spread: float = float(cfg.get(\"spread\", 0.0))\n        self.fee_pct: float = float(cfg.get(\"fee_pct\", 0.0))\n        self.funding_model: str = str(cfg.get(\"funding_model\",\"8h_to_5m\"))\n        self.random_start: bool = bool(cfg.get(\"random_start\", True))\n        self.episode_len: int = int(cfg.get(\"episode_len\", 2400))\n        self.if_train: bool = bool(cfg.get(\"if_train\", True))\n        self.seed_value: int = int(cfg.get(\"seed\", 1337))\n        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(28,), dtype=np.float32)\n        self.action_space = spaces.MultiDiscrete(np.array([4,4,5], dtype=np.int64))\n        self.rng = np.random.default_rng(self.seed_value)\n        self.ptr: int = 0\n        self.start_ptr: int = 0\n        self.end_ptr: int = min(self.episode_len, len(self.data)-1)\n        self.equity: float = self.initial_equity\n        self.pos_dir: int = 0\n        self.lev_idx: int = 0\n        self.stop_idx: int = 0\n        self.entry_px: Optional[float] = None\n\n    def _encode_obs(self, idx: int) -> np.ndarray:\n        idx = max(0, min(idx, len(self.data)-1))\n        row = self.data.iloc[idx]\n        return row[TOKEN_COLUMNS].to_numpy(dtype=np.float32)\n\n    def _legal_triples(self, row: pd.Series, blind: bool) -> List[Tuple[int,int,int]]:\n        legal: List[Tuple[int,int,int]] = []\n        if blind:\n            legal.append((0,0,0))\n            legal.append((3,0,0))\n            return legal\n        legal.append((0,0,0))\n        legal.append((3,0,0))\n        if self.pos_dir == 0:\n            for lev in range(len(self.leverage_slots)):\n                for stp in range(len(self.stop_slots)):\n                    legal.append((1,lev,stp))\n                    legal.append((2,lev,stp))\n        elif self.pos_dir > 0:\n            for lev in range(self.lev_idx, len(self.leverage_slots)):\n                for stp in range(0, self.stop_idx+1):\n                    legal.append((1,lev,stp))\n        else:\n            for lev in range(self.lev_idx, len(self.leverage_slots)):\n                for stp in range(0, self.stop_idx+1):\n                    legal.append((2,lev,stp))\n        return legal\n\n    def _coerce_to_legal(self, action: np.ndarray, legal: List[Tuple[int,int,int]]) -> Tuple[int,int,int]:\n        a = (int(action[0]), int(action[1]), int(action[2]))\n        if a in legal:\n            return a\n        return (0,0,0)\n\n    def _position_notional(self, ref_equity: float) -> float:\n        if self.pos_dir == 0:\n            return 0.0\n        lev = self.leverage_slots[self.lev_idx]\n        return float(ref_equity) * float(lev)\n\n    def _funding_per_bar(self, eight_hour_rate: float) -> float:\n        if self.funding_model != \"8h_to_5m\":\n            raise ValueError(\"Only funding_model='8h_to_5m' is supported.\")\n        return float(eight_hour_rate) / 96.0\n\n    def reset(self):\n        n = len(self.data)\n        if self.random_start and self.if_train:\n            max_start = max(0, n - (self.episode_len + 1))\n            self.start_ptr = int(self.rng.integers(0, max_start+1)) if max_start > 0 else 0\n        else:\n            self.start_ptr = 0\n        self.ptr = self.start_ptr\n        self.end_ptr = min(self.start_ptr + self.episode_len, n-1)\n        self.equity = self.initial_equity\n        self.pos_dir = 0\n        self.lev_idx = 0\n        self.stop_idx = 0\n        self.entry_px = None\n        row = self.data.iloc[self.ptr]\n        if (row[\"fund_ts\"] > row[\"ts\"]) or (row[\"liq_ts\"] > row[\"ts\"]) or (row[\"basis_ts\"] > row[\"ts\"]):\n            raise AssertionError(\"Slow-feed timestamp later than bar time at reset.\")\n        return self._encode_obs(self.ptr)\n\n    def step(self, action: np.ndarray):\n        row = self.data.iloc[self.ptr]\n        prev_idx = max(0, self.ptr-1)\n        prev_row = self.data.iloc[prev_idx]\n        blind = (int(row.get(\"feed_alive\", 1)) == 0) or (int(row.get(\"depth_available\", 1)) == 0)\n        legal = self._legal_triples(row, blind)\n        op, lev_sel, stp_sel = self._coerce_to_legal(action, legal)\n        prev_equity = self.equity\n        cash_delta = 0.0\n        if self.pos_dir != 0:\n            px_prev = float(prev_row[\"px\"])\n            px_now  = float(row[\"px\"])\n            if px_prev <= 0 or px_now <= 0:\n                raise AssertionError(\"Non-positive price encountered.\")\n            ret = (px_now / px_prev) - 1.0\n            notional = self._position_notional(prev_equity)\n            price_pnl = float(self.pos_dir) * notional * ret\n            cash_delta += price_pnl\n            eight_hr = float(row[\"fund_vwap\"])\n            per_bar = self._funding_per_bar(eight_hr)\n            carry = -abs(notional) * per_bar if self.pos_dir > 0 else +abs(notional) * per_bar\n            cash_delta += carry\n        old_notional = self._position_notional(prev_equity)\n        if op == 3:\n            new_pos_dir, new_lev_idx, new_stop_idx = 0, 0, 0\n        elif op == 1:\n            new_pos_dir = +1 if self.pos_dir == 0 else self.pos_dir\n            new_lev_idx = lev_sel if self.pos_dir == 0 else max(self.lev_idx, lev_sel)\n            new_stop_idx = stp_sel if self.pos_dir == 0 else min(self.stop_idx, stp_sel)\n        elif op == 2:\n            new_pos_dir = -1 if self.pos_dir == 0 else self.pos_dir\n            new_lev_idx = lev_sel if self.pos_dir == 0 else max(self.lev_idx, lev_sel)\n            new_stop_idx = stp_sel if self.pos_dir == 0 else min(self.stop_idx, stp_sel)\n        else:\n            new_pos_dir, new_lev_idx, new_stop_idx = self.pos_dir, self.lev_idx, self.stop_idx\n        if self.pos_dir > 0 and op == 2:\n            new_pos_dir, new_lev_idx, new_stop_idx = self.pos_dir, self.lev_idx, self.stop_idx\n        if self.pos_dir < 0 and op == 1:\n            new_pos_dir, new_lev_idx, new_stop_idx = self.pos_dir, self.lev_idx, self.stop_idx\n        new_notional = 0.0 if new_pos_dir == 0 else float(prev_equity) * float(self.leverage_slots[new_lev_idx])\n        trade_delta = new_notional - old_notional\n        fill_cost = abs(trade_delta) * (abs(self.fee_pct) + abs(self.spread))\n        cash_delta -= fill_cost\n        self.equity = float(self.equity + cash_delta)\n        self.pos_dir = new_pos_dir\n        self.lev_idx = int(new_lev_idx)\n        self.stop_idx = int(new_stop_idx)\n        self.entry_px = float(row[\"px\"]) if (self.pos_dir != 0 and (op in (1,2) or old_notional == 0)) else self.entry_px\n        r_unit = max(prev_equity * self.risk_cap_pct, 1e-6)\n        reward = cash_delta / r_unit\n        reward = float(np.clip(reward, self.reward_clip[0], self.reward_clip[1]))\n        self.ptr += 1\n        done = bool(self.ptr >= self.end_ptr or self.ptr >= (len(self.data)-1))\n        if self.ptr < len(self.data):\n            nrow = self.data.iloc[self.ptr]\n            if (nrow[\"fund_ts\"] > nrow[\"ts\"]) or (nrow[\"liq_ts\"] > nrow[\"ts\"]) or (nrow[\"basis_ts\"] > nrow[\"ts\"]):\n                raise AssertionError(\"Slow-feed timestamp later than bar time at next step.\")\n        obs = self._encode_obs(self.ptr if self.ptr < len(self.data) else len(self.data)-1)\n        info = {\"pnl\": cash_delta, \"equity\": self.equity, \"prev_equity\": prev_equity}\n        return obs, reward, done, info\n\n    def legal_actions_flat(self) -> List[int]:\n        row = self.data.iloc[self.ptr]\n        blind = (int(row.get(\"feed_alive\",1)) == 0) or (int(row.get(\"depth_available\",1)) == 0)\n        triples = self._legal_triples(row, blind)\n        return [_flatten_idx(*t) for t in triples]\n\nclass SB3Adapter(gym.Env):\n    metadata = {\"render_modes\": []}\n    def __init__(self, base_env: SwingTradingEnv):\n        super().__init__()\n        self.base = base_env\n        self.action_space = self.base.action_space\n        self.observation_space = self.base.observation_space\n    def reset(self, *, seed: Optional[int] = None, options: Optional[dict] = None):\n        if seed is not None:\n            self.base.rng = np.random.default_rng(int(seed))\n        obs = self.base.reset()\n        info = {}\n        return obs, info\n    def step(self, action):\n        obs, reward, done, info = self.base.step(action)\n        terminated = bool(done)\n        truncated = False\n        return obs, float(reward), terminated, truncated, info\n\ndef _flatten_idx(op: int, lev: int, stp: int) -> int:\n    return op * 20 + lev * 5 + stp\n\ndef _unflatten_idx(idx: int) -> Tuple[int,int,int]:\n    op = idx // 20\n    r = idx % 20\n    lev = r // 5\n    stp = r % 5\n    return op, lev, stp\n\nclass FlattenActionWrapper(gym.Wrapper):\n    def __init__(self, env: gym.Env):\n        super().__init__(env)\n        assert isinstance(env.action_space, spaces.MultiDiscrete), \"Underlying env must be MultiDiscrete.\"\n        self.action_space = spaces.Discrete(80)\n    def reset(self, **kwargs):\n        return self.env.reset(**kwargs)\n    def step(self, action_idx: int):\n        op, lev, stp = _unflatten_idx(int(action_idx))\n        return self.env.step(np.array([op,lev,stp], dtype=np.int64))\n    def get_action_mask_80(self) -> np.ndarray:\n        legal = self.env.base.legal_actions_flat() if isinstance(self.env, SB3Adapter) else self.env.legal_actions_flat()\n        mask = np.zeros(80, dtype=bool)\n        mask[np.array(legal, dtype=int)] = True\n        return mask\n"v2.py","w", encoding="utf-8") as f:
    f.write("\n# \u2500\u2500 swing_trading_env_v2.py \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nfrom __future__ import annotations\nimport numpy as np\nimport pandas as pd\nfrom typing import Tuple, List, Optional, Dict, Any\nimport warnings\n\nimport gymnasium as gym\nfrom gymnasium import spaces\n\nTOKEN_COLUMNS: List[str] = [\n    \"log_ret_open\",\"log_ret_high\",\"log_ret_low\",\"log_ret_close\",\"range_frac\",\n    \"log_volume\",\"volume_zscore\",\"depth_10bp\",\"bid_ask_imbalance\",\n    \"fund_binance\",\"fund_bybit\",\"fund_okx\",\"fund_vwap\",\"fund_zscore\",\n    \"fund_spread_bin_byb\",\"fund_spread_width\",\"fund_flip\",\"fund_accum_24h\",\n    \"perp_spot_basis\",\n    \"liq_buy_1d\",\"liq_buy_6h\",\"liq_buy_4h\",\"liq_sell_1d\",\"liq_sell_6h\",\"liq_sell_4h\",\n    \"vol_of_vol\",\"sin_time\",\"cos_time\",\n]\n\ndef _assert_token_contract(df: pd.DataFrame):\n    required = [\"ts\",\"px\",\"fund_vwap\",\"fund_ts\",\"liq_ts\",\"basis_ts\",\"feed_alive\"]\n    missing = [c for c in required if c not in df.columns]\n    if missing:\n        raise ValueError(f\"Data missing required columns: {missing}\")\n    for c in TOKEN_COLUMNS:\n        if c not in df.columns:\n            warnings.warn(f\"Column '{c}' missing in data; filling zeros (development backfill).\")\n            df[c] = 0.0\n    if not np.issubdtype(df[\"ts\"].dtype, np.datetime64):\n        df[\"ts\"] = pd.to_datetime(df[\"ts\"], utc=True)\n    else:\n        df[\"ts\"] = df[\"ts\"].dt.tz_localize(\"UTC\") if df[\"ts\"].dt.tz is None else df[\"ts\"].dt.tz_convert(\"UTC\")\n    if not df[\"ts\"].is_monotonic_increasing:\n        raise AssertionError(\"Timestamps must be strictly increasing.\")\n    ms = (df[\"ts\"].astype(\"int64\") // 1_000_000).to_numpy()\n    if np.any(ms % 300_000 != 0):\n        raise AssertionError(\"Timestamps must align to 5-minute anchors (:00,:05,:10,...) UTC.\")\n    for c in (\"fund_ts\",\"liq_ts\",\"basis_ts\"):\n        if not np.issubdtype(df[c].dtype, np.datetime64):\n            df[c] = pd.to_datetime(df[c], utc=True)\n        else:\n            df[c] = df[c].dt.tz_localize(\"UTC\") if df[c].dt.tz is None else df[c].dt.tz_convert(\"UTC\")\n    df[\"feed_alive\"] = df[\"feed_alive\"].astype(int)\n\ndef _flatten_idx(op: int, lev: int, stp: int) -> int:\n    return op * 20 + lev * 5 + stp\n\ndef _unflatten_idx(idx: int) -> Tuple[int,int,int]:\n    op = idx // 20\n    r = idx % 20\n    lev = r // 5\n    stp = r % 5\n    return op, lev, stp\n\nclass SwingTradingEnv:\n    metadata: Dict[str, Any] = {\"render.modes\": []}\n    def __init__(self, cfg: Dict[str, Any]):\n        self.cfg = cfg.copy()\n        self.data: pd.DataFrame = cfg[\"data\"].copy()\n        _assert_token_contract(self.data)\n        self.initial_equity: float = float(cfg.get(\"initial_equity\", 10_000.0))\n        self.risk_cap_pct: float = float(cfg.get(\"risk_cap_pct\", 0.05))\n        self.reward_clip: Tuple[float,float] = tuple(cfg.get(\"reward_clip\", (-1.0, 1.0)))\n        self.max_leverage: int = int(cfg.get(\"max_leverage\", 5))\n        self.leverage_slots: List[int] = list(cfg.get(\"leverage_slots\",[1,2,3,5]))\n        self.stop_slots: List[float] = list(cfg.get(\"stop_slots\",[0.001,0.002,0.003,0.005,0.008]))\n        self.spread: float = float(cfg.get(\"spread\", 0.0))\n        self.fee_pct: float = float(cfg.get(\"fee_pct\", 0.0))\n        self.funding_model: str = str(cfg.get(\"funding_model\",\"8h_to_5m\"))\n        self.random_start: bool = bool(cfg.get(\"random_start\", True))\n        self.episode_len: int = int(cfg.get(\"episode_len\", 2400))\n        self.if_train: bool = bool(cfg.get(\"if_train\", True))\n        self.seed_value: int = int(cfg.get(\"seed\", 1337))\n        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(28,), dtype=np.float32)\n        self.action_space = spaces.MultiDiscrete(np.array([4,4,5], dtype=np.int64))\n        self.rng = np.random.default_rng(self.seed_value)\n        self.ptr: int = 0\n        self.start_ptr: int = 0\n        self.end_ptr: int = min(self.episode_len, len(self.data)-1)\n        self.equity: float = self.initial_equity\n        self.pos_dir: int = 0\n        self.lev_idx: int = 0\n        self.stop_idx: int = 0\n        self.entry_px: Optional[float] = None\n\n    def _encode_obs(self, idx: int) -> np.ndarray:\n        idx = max(0, min(idx, len(self.data)-1))\n        row = self.data.iloc[idx]\n        return row[TOKEN_COLUMNS].to_numpy(dtype=np.float32)\n\n    def _legal_triples(self, row: pd.Series, blind: bool) -> List[Tuple[int,int,int]]:\n        legal: List[Tuple[int,int,int]] = []\n        if blind:\n            legal.append((0,0,0))\n            legal.append((3,0,0))\n            return legal\n        legal.append((0,0,0))\n        legal.append((3,0,0))\n        if self.pos_dir == 0:\n            for lev in range(len(self.leverage_slots)):\n                for stp in range(len(self.stop_slots)):\n                    legal.append((1,lev,stp))\n                    legal.append((2,lev,stp))\n        elif self.pos_dir > 0:\n            for lev in range(self.lev_idx, len(self.leverage_slots)):\n                for stp in range(0, self.stop_idx+1):\n                    legal.append((1,lev,stp))\n        else:\n            for lev in range(self.lev_idx, len(self.leverage_slots)):\n                for stp in range(0, self.stop_idx+1):\n                    legal.append((2,lev,stp))\n        return legal\n\n    def _coerce_to_legal(self, action: np.ndarray, legal: List[Tuple[int,int,int]]) -> Tuple[int,int,int]:\n        a = (int(action[0]), int(action[1]), int(action[2]))\n        if a in legal:\n            return a\n        return (0,0,0)\n\n    def _position_notional(self, ref_equity: float) -> float:\n        if self.pos_dir == 0:\n            return 0.0\n        lev = self.leverage_slots[self.lev_idx]\n        return float(ref_equity) * float(lev)\n\n    def _funding_per_bar(self, eight_hour_rate: float) -> float:\n        if self.funding_model != \"8h_to_5m\":\n            raise ValueError(\"Only funding_model='8h_to_5m' is supported.\")\n        return float(eight_hour_rate) / 96.0\n\n    def reset(self):\n        n = len(self.data)\n        if self.random_start and self.if_train:\n            max_start = max(0, n - (self.episode_len + 1))\n            self.start_ptr = int(self.rng.integers(0, max_start+1)) if max_start > 0 else 0\n        else:\n            self.start_ptr = 0\n        self.ptr = self.start_ptr\n        self.end_ptr = min(self.start_ptr + self.episode_len, n-1)\n        self.equity = self.initial_equity\n        self.pos_dir = 0\n        self.lev_idx = 0\n        self.stop_idx = 0\n        self.entry_px = None\n        row = self.data.iloc[self.ptr]\n        if (row[\"fund_ts\"] > row[\"ts\"]) or (row[\"liq_ts\"] > row[\"ts\"]) or (row[\"basis_ts\"] > row[\"ts\"]):\n            raise AssertionError(\"Slow-feed timestamp later than bar time at reset.\")\n        return self._encode_obs(self.ptr)\n\n    def step(self, action: np.ndarray):\n        row = self.data.iloc[self.ptr]\n        prev_idx = max(0, self.ptr-1)\n        prev_row = self.data.iloc[prev_idx]\n        blind = (int(row.get(\"feed_alive\", 1)) == 0) or (int(row.get(\"depth_available\", 1)) == 0)\n        legal = self._legal_triples(row, blind)\n        op, lev_sel, stp_sel = self._coerce_to_legal(action, legal)\n        prev_equity = self.equity\n        cash_delta = 0.0\n        if self.pos_dir != 0:\n            px_prev = float(prev_row[\"px\"])\n            px_now  = float(row[\"px\"])\n            if px_prev <= 0 or px_now <= 0:\n                raise AssertionError(\"Non-positive price encountered.\")\n            ret = (px_now / px_prev) - 1.0\n            notional = self._position_notional(prev_equity)\n            price_pnl = float(self.pos_dir) * notional * ret\n            cash_delta += price_pnl\n            eight_hr = float(row[\"fund_vwap\"])\n            per_bar = self._funding_per_bar(eight_hr)\n            carry = -abs(notional) * per_bar if self.pos_dir > 0 else +abs(notional) * per_bar\n            cash_delta += carry\n        old_notional = self._position_notional(prev_equity)\n        if op == 3:\n            new_pos_dir, new_lev_idx, new_stop_idx = 0, 0, 0\n        elif op == 1:\n            new_pos_dir = +1 if self.pos_dir == 0 else self.pos_dir\n            new_lev_idx = lev_sel if self.pos_dir == 0 else max(self.lev_idx, lev_sel)\n            new_stop_idx = stp_sel if self.pos_dir == 0 else min(self.stop_idx, stp_sel)\n        elif op == 2:\n            new_pos_dir = -1 if self.pos_dir == 0 else self.pos_dir\n            new_lev_idx = lev_sel if self.pos_dir == 0 else max(self.lev_idx, lev_sel)\n            new_stop_idx = stp_sel if self.pos_dir == 0 else min(self.stop_idx, stp_sel)\n        else:\n            new_pos_dir, new_lev_idx, new_stop_idx = self.pos_dir, self.lev_idx, self.stop_idx\n        if self.pos_dir > 0 and op == 2:\n            new_pos_dir, new_lev_idx, new_stop_idx = self.pos_dir, self.lev_idx, self.stop_idx\n        if self.pos_dir < 0 and op == 1:\n            new_pos_dir, new_lev_idx, new_stop_idx = self.pos_dir, self.lev_idx, self.stop_idx\n        new_notional = 0.0 if new_pos_dir == 0 else float(prev_equity) * float(self.leverage_slots[new_lev_idx])\n        trade_delta = new_notional - old_notional\n        fill_cost = abs(trade_delta) * (abs(self.fee_pct) + abs(self.spread))\n        cash_delta -= fill_cost\n        self.equity = float(self.equity + cash_delta)\n        self.pos_dir = new_pos_dir\n        self.lev_idx = int(new_lev_idx)\n        self.stop_idx = int(new_stop_idx)\n        self.entry_px = float(row[\"px\"]) if (self.pos_dir != 0 and (op in (1,2) or old_notional == 0)) else self.entry_px\n        r_unit = max(prev_equity * self.risk_cap_pct, 1e-6)\n        reward = cash_delta / r_unit\n        reward = float(np.clip(reward, self.reward_clip[0], self.reward_clip[1]))\n        self.ptr += 1\n        done = bool(self.ptr >= self.end_ptr or self.ptr >= (len(self.data)-1))\n        if self.ptr < len(self.data):\n            nrow = self.data.iloc[self.ptr]\n            if (nrow[\"fund_ts\"] > nrow[\"ts\"]) or (nrow[\"liq_ts\"] > nrow[\"ts\"]) or (nrow[\"basis_ts\"] > nrow[\"ts\"]):\n                raise AssertionError(\"Slow-feed timestamp later than bar time at next step.\")\n        obs = self._encode_obs(self.ptr if self.ptr < len(self.data) else len(self.data)-1)\n        info = {\"pnl\": cash_delta, \"equity\": self.equity, \"prev_equity\": prev_equity}\n        return obs, reward, done, info\n\n    def legal_actions_flat(self) -> List[int]:\n        row = self.data.iloc[self.ptr]\n        blind = (int(row.get(\"feed_alive\",1)) == 0) or (int(row.get(\"depth_available\",1)) == 0)\n        triples = self._legal_triples(row, blind)\n        return [_flatten_idx(*t) for t in triples]\n\nclass SB3Adapter(gym.Env):\n    metadata = {\"render_modes\": []}\n    def __init__(self, base_env: SwingTradingEnv):\n        super().__init__()\n        self.base = base_env\n        self.action_space = self.base.action_space\n        self.observation_space = self.base.observation_space\n    def reset(self, *, seed: Optional[int] = None, options: Optional[dict] = None):\n        if seed is not None:\n            self.base.rng = np.random.default_rng(int(seed))\n        obs = self.base.reset()\n        info = {}\n        return obs, info\n    def step(self, action):\n        obs, reward, done, info = self.base.step(action)\n        terminated = bool(done)\n        truncated = False\n        return obs, float(reward), terminated, truncated, info\n\ndef _flatten_idx(op: int, lev: int, stp: int) -> int:\n    return op * 20 + lev * 5 + stp\n\ndef _unflatten_idx(idx: int) -> Tuple[int,int,int]:\n    op = idx // 20\n    r = idx % 20\n    lev = r // 5\n    stp = r % 5\n    return op, lev, stp\n\nclass FlattenActionWrapper(gym.Wrapper):\n    def __init__(self, env: gym.Env):\n        super().__init__(env)\n        assert isinstance(env.action_space, spaces.MultiDiscrete), \"Underlying env must be MultiDiscrete.\"\n        self.action_space = spaces.Discrete(80)\n    def reset(self, **kwargs):\n        return self.env.reset(**kwargs)\n    def step(self, action_idx: int):\n        op, lev, stp = _unflatten_idx(int(action_idx))\n        return self.env.step(np.array([op,lev,stp], dtype=np.int64))\n    def get_action_mask_80(self) -> np.ndarray:\n        legal = self.env.base.legal_actions_flat() if isinstance(self.env, SB3Adapter) else self.env.legal_actions_flat()\n        mask = np.zeros(80, dtype=bool)\n        mask[np.array(legal, dtype=int)] = True\n        return mask\n")

## Data Harness (Phase‑4 style)

In [ ]:
# Phase‑4 style data harness: load token + meta, join, enforce schema and leak checks

import json, os, numpy as np, pandas as pd
from pathlib import Path

# Load token & meta
print("Loading:", TOKEN_ALL)
df_tok = pd.read_parquet(TOKEN_ALL)
print("Loading:", META_PATH)
df_meta = pd.read_parquet(META_PATH)

# Ensure UTC ts and join
def as_utc(s):
    s = pd.to_datetime(s, utc=True)
    return s

df_tok["ts"] = as_utc(df_tok["ts"])
df_meta["ts"] = as_utc(df_meta["ts"])

# Select meta columns we need
needed_meta = ["ts","feed_alive","fund_ts","liq_ts","basis_ts"]
for c in needed_meta:
    if c not in df_meta.columns:
        raise ValueError(f"Meta missing required column: {c}")

df = pd.merge_asof(
    df_tok.sort_values("ts"),
    df_meta[needed_meta].sort_values("ts"),
    on="ts", direction="nearest", tolerance=pd.Timedelta("0s")
)

# Basic NA handling for control columns
if df["feed_alive"].isna().any():
    df["feed_alive"] = df["feed_alive"].fillna(0).astype(int)
for c in ["fund_ts","liq_ts","basis_ts"]:
    if df[c].isna().any():
        # Conservative: if missing slow ts, mark as stale (set to very old)
        df.loc[df[c].isna(), c] = df.loc[df[c].isna(), "ts"] - pd.Timedelta(days=365*50)
        df[c] = pd.to_datetime(df[c], utc=True)

# Ensure px exists; if not, try to synthesize from log_ret_close (fallback is not ideal)
if "px" not in df.columns:
    raise ValueError("Token frame must include 'px' (mark price) for PnL computation.")

# Token v1 column enforcement: cast to float32, create missing as zeros
TOKEN_COLUMNS = [
    "log_ret_open","log_ret_high","log_ret_low","log_ret_close","range_frac",
    "log_volume","volume_zscore","depth_10bp","bid_ask_imbalance",
    "fund_binance","fund_bybit","fund_okx","fund_vwap","fund_zscore",
    "fund_spread_bin_byb","fund_spread_width","fund_flip","fund_accum_24h",
    "perp_spot_basis",
    "liq_buy_1d","liq_buy_6h","liq_buy_4h","liq_sell_1d","liq_sell_6h","liq_sell_4h",
    "vol_of_vol","sin_time","cos_time",
]
for c in TOKEN_COLUMNS:
    if c not in df.columns:
        print(f"[warn] missing {c}, filling zeros")
        df[c] = 0.0
for c in TOKEN_COLUMNS + ["px"]:
    df[c] = df[c].astype("float32")

# Monotonic & 5m anchors
if not df["ts"].is_monotonic_increasing:
    raise AssertionError("Timestamps must be strictly increasing.")
ms = (df["ts"].astype("int64") // 1_000_000).to_numpy()
if np.any(ms % 300_000 != 0):
    raise AssertionError("Timestamps must align to 5-minute anchors.")

# NaN sweep on token columns
nan_cols = [c for c in TOKEN_COLUMNS if df[c].isna().any()]
if nan_cols:
    raise AssertionError(f"NaNs detected in token columns: {nan_cols}")

# Leak audit: all slow feeds must be <= bar time
for c in ["fund_ts","liq_ts","basis_ts"]:
    bad = (df[c] > df["ts"]).sum()
    if bad:
        raise AssertionError(f"Leak detected: {bad} rows have {c} later than ts.")

# Basis envelope (robust): flag extremely outlying basis; fail if more than 0.1% of rows
if "perp_spot_basis" in df.columns:
    b = df["perp_spot_basis"].astype("float64")
    med = float(np.median(b))
    mad = float(np.median(np.abs(b - med))) or 1e-8
    z = 0.6745 * (b - med) / mad
    frac_bad = float((np.abs(z) > 12).mean())  # very tight envelope
    print(f"Basis envelope: |z|>12 fraction = {frac_bad:.6f}")
    if frac_bad > 0.001:
        raise AssertionError("Basis envelope breach beyond tolerance.")

print("Harness checks passed.")
# Keep slim copy for env (only columns we use at step time)
keep_cols = ["ts","px","feed_alive","fund_ts","liq_ts","basis_ts"] + TOKEN_COLUMNS
core_df = df[keep_cols].copy()
core_df = core_df.reset_index(drop=True)

print(core_df.dtypes.head())
print("Rows:", len(core_df), "From:", core_df['ts'].min(), "To:", core_df['ts'].max())

# Expose to later cells
CORE_DF = core_df  # global var in notebook

## Build Walk‑Forward Splits

In [ ]:
# Walk-forward splits: Train=2021, Val=2022, Test=2023, Tail=2024+ (when present)
import pandas as pd

def split_by_year(df):
    y = df["ts"].dt.year
    s = {
        "train_2021": df[(y == 2021)],
        "val_2022":   df[(y == 2022)],
        "test_2023":  df[(y == 2023)],
        "tail_2024p": df[(y >= 2024)],
    }
    return s

SPLIT = split_by_year(CORE_DF)

# Fallback if 2021..2023 not present: use last 90d for quick experimentation
if len(SPLIT["train_2021"]) < 1000 or len(SPLIT["val_2022"]) < 1000 or len(SPLIT["test_2023"]) < 1000:
    print("Yearly splits too small; falling back to last90d parquet for a runnable demo.")
    df_tok = pd.read_parquet(TOKEN_L90)
    df_tok["ts"] = pd.to_datetime(df_tok["ts"], utc=True)
    df_meta = pd.read_parquet(META_PATH)[["ts","feed_alive","fund_ts","liq_ts","basis_ts"]]
    df_meta["ts"] = pd.to_datetime(df_meta["ts"], utc=True)
    df = pd.merge_asof(df_tok.sort_values("ts"), df_meta.sort_values("ts"), on="ts", direction="nearest", tolerance=pd.Timedelta("0s"))
    # Ensure columns
    for c in ["feed_alive","fund_ts","liq_ts","basis_ts"]:
        if c not in df.columns:
            raise ValueError(f"Meta join missing {c}")
    # Token v1 fixes
    for c in CORE_DF.columns:
        if c not in df.columns and c not in ("ts",):
            df[c] = 0.0
    CORE_DF_L90 = df[CORE_DF.columns].copy()
    y = CORE_DF_L90["ts"]
    # 60d/30d split
    cutoff_val = y.max() - pd.Timedelta(days=30)
    cutoff_trn = cutoff_val - pd.Timedelta(days=60)
    SPLIT = {
        "train_2021": CORE_DF_L90[(y >= cutoff_trn) & (y < cutoff_val)].copy(),
        "val_2022":   CORE_DF_L90[(y >= cutoff_val)].copy().iloc[: int(0.6 * len(CORE_DF_L90[(y >= cutoff_val)]))].copy(),
        "test_2023":  CORE_DF_L90[(y >= cutoff_val)].copy().iloc[int(0.6 * len(CORE_DF_L90[(y >= cutoff_val)])) :].copy(),
        "tail_2024p": CORE_DF_L90.tail(0).copy(),
    }

for k,v in SPLIT.items():
    print(k, "rows:", len(v), "range:", v["ts"].min(), "→", v["ts"].max())

## Train & Evaluate (MaskablePPO, action masks respected)

In [ ]:
# Train MaskablePPO on Train(2021) and evaluate on Val(2022) and Test(2023)

import numpy as np, pandas as pd
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
from stable_baselines3.common.vec_env import DummyVecEnv
from swing_trading_env_v2 import SwingTradingEnv, SB3Adapter, FlattenActionWrapper

def make_env_from_df(df, if_train=True, episode_len=1280, seed=1337):
    df = df.copy()
    for col in ["ts", "fund_ts", "liq_ts", "basis_ts"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], utc=True).dt.tz_localize(None)
    cfg = {
        "data": df.reset_index(drop=True),
        "initial_equity": 10_000.0,
        "risk_cap_pct": 0.05,
        "reward_clip": (-1.0, 1.0),
        "max_leverage": 5,
        "leverage_slots": [1,2,3,5],
        "stop_slots": [0.001, 0.002, 0.003, 0.005, 0.008],
        "spread": 0.0005,
        "fee_pct": 0.004,
        "funding_model": "8h_to_5m",
        "random_start": if_train,
        "episode_len": min(episode_len, max(8, len(df)-2)),
        "if_train": if_train,
        "seed": seed
    }
    base = SwingTradingEnv(cfg)
    sb3_env = SB3Adapter(base)
    flat_env = FlattenActionWrapper(sb3_env)
    masked_env = ActionMasker(flat_env, flat_env.get_action_mask_80)
    return masked_env

def vec_env_from_df(df, n_env=8, if_train=True, episode_len=1280, seed=1337):
    return DummyVecEnv([lambda i=i: make_env_from_df(df, if_train=if_train, episode_len=episode_len, seed=seed+i) for i in range(n_env)])

def evaluate_model_on_df(model, df):
    env = vec_env_from_df(df, n_env=1, if_train=False, episode_len=len(df)-2, seed=999)
    obs, info = env.reset()
    done = False
    records = []
    while True:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        info0 = info[0] if isinstance(info, list) else info
        # Pull equity info from underlying base env
        # Using SB3Adapter->FlattenActionWrapper->ActionMasker chain: info already carries {'equity','prev_equity','pnl'}
        eq = float(info0.get("equity", np.nan))
        peq = float(info0.get("prev_equity", np.nan))
        pnl = float(info0.get("pnl", np.nan))
        records.append((eq, peq, pnl, float(reward[0]) if isinstance(reward, np.ndarray) else float(reward)))
        if bool(terminated[0] if isinstance(terminated, np.ndarray) else terminated) or bool(truncated[0] if isinstance(truncated, np.ndarray) else truncated):
            break
    env.close()
    df_out = pd.DataFrame(records, columns=["equity","prev_equity","pnl","reward"])
    df_out["ret"] = (df_out["equity"] - df_out["prev_equity"]) / df_out["prev_equity"].replace(0, np.nan)
    return df_out.dropna().reset_index(drop=True)

def sharpe_from_returns(r, bars_per_day=288):
    mu, sd = float(np.mean(r)), float(np.std(r) + 1e-12)
    daily = (mu / sd) * np.sqrt(bars_per_day) if sd > 0 else 0.0
    return daily

def max_drawdown(equity):
    peak = np.maximum.accumulate(equity)
    dd = (equity - peak) / np.where(peak==0, np.nan, peak)
    return float(np.nanmin(dd))

def calmar(equity, bars_per_year=105120): # 5m bars ≈ 365*24*12
    ret = equity[-1] / equity[0] - 1.0 if equity[0] > 0 else 0.0
    years = len(equity) / bars_per_year
    ann = (1.0 + ret) ** (1.0 / max(years, 1e-6)) - 1.0 if years > 0 else ret
    mdd = abs(max_drawdown(equity)) or 1e-12
    return float(ann / mdd)

def turnover_proxy(pnl, prev_eq):
    # Rough proxy: sum of absolute pnl caused by fills vs equity (requires separable accounting; here approximate)
    # We approximate turnover by counting action-induced notional deltas via reward spikes (<-1 or >+1) is noisy;
    # For simplicity, compute |pnl| / prev_eq aggregated.
    x = np.sum(np.abs(pnl) / np.maximum(prev_eq, 1e-6))
    return float(x)

def stop_hit_rate_proxy(reward_series):
    # Proxy: fraction of steps that hit the negative clamp (i.e., full risk‑unit loss crystallized in one bar)
    r = np.asarray(reward_series, dtype=float)
    return float((r <= -1.0 + 1e-9).mean())

# --- Build envs
train_df = SPLIT["train_2021"]
val_df   = SPLIT["val_2022"]
test_df  = SPLIT["test_2023"]

print("Train rows:", len(train_df), "Val rows:", len(val_df), "Test rows:", len(test_df))

train_env = vec_env_from_df(train_df, n_env=8, if_train=True, episode_len=1280, seed=1337)

model = MaskablePPO(
    policy="MlpPolicy",
    env=train_env,
    verbose=1,
    n_steps=128,
    batch_size=128,
    ent_coef=0.02,
    vf_coef=0.50,
    clip_range=0.20,
    learning_rate=3e-4,
    gamma=0.99
)

TOTAL_TIMESTEPS = 200_000  # Adjust in Colab if needed
model.learn(total_timesteps=TOTAL_TIMESTEPS)

# Evaluate
val_res  = evaluate_model_on_df(model, val_df)
test_res = evaluate_model_on_df(model, test_df)

def summarize(df_res, label):
    sh = sharpe_from_returns(df_res["ret"].to_numpy())
    mdd = max_drawdown(df_res["equity"].to_numpy())
    cal = calmar(df_res["equity"].to_numpy())
    to  = turnover_proxy(df_res["pnl"].to_numpy(), df_res["prev_equity"].to_numpy())
    stopr = stop_hit_rate_proxy(df_res["reward"].to_numpy())
    print(f"[{label}] Sharpe/daily={sh:.3f}  MaxDD={mdd:.3%}  Calmar={cal:.3f}  Turnover≈{to:.3f}  StopHitProxy={stopr:.3%}")
    return {"label":label,"sharpe_daily":sh,"max_drawdown":mdd,"calmar":cal,"turnover_proxy":to,"stop_hit_proxy":stopr}

metrics = [summarize(val_res, "VAL_2022"), summarize(test_res, "TEST_2023")]
metrics_df = pd.DataFrame(metrics)
metrics_df

## Save Artifacts to Drive

In [ ]:
# Save artifacts (model + metrics) back to Drive (optional)
import os
os.makedirs("/content/drive/MyDrive/Crypto_Data/runs", exist_ok=True)
model_path = "/content/drive/MyDrive/Crypto_Data/runs/maskableppo_2021.zip"
metrics_path = "/content/drive/MyDrive/Crypto_Data/runs/walkforward_metrics.csv"
model.save(model_path)
metrics_df.to_csv(metrics_path, index=False)
print("Saved:", model_path)
print("Saved:", metrics_path)